# TFM — Detección de Anomalías en ECG con Big Data
## Visualización de Resultados (Anexo F)
**Andrea Ríos López — Master Big Data & Advanced Analytics**

Este notebook implementa la capa de visualización del TFM, operando sobre los datos
almacenados en Snowflake. Genera tres vistas analíticas:

- **Vista 1**: Total de anomalías detectadas por modelo
- **Vista 2**: Tasa de detección por diagnóstico (Isolation Forest — Top 10)
- **Vista 3**: Señal ECG individual con error de reconstrucción punto a punto (LSTM)

---
## F.1 — Instalación de dependencias

In [ ]:
%pip install snowflake-connector-python matplotlib seaborn pandas wfdb torch --quiet

In [ ]:
dbutils.library.restartPython()

## F.2 — Configuración: Snowflake y AWS S3

In [ ]:
import snowflake.connector
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io, os, pickle, wfdb
import torch
import torch.nn as nn
from datetime import datetime

# ── Conexión Snowflake ───────────────────────────────────
conn = snowflake.connector.connect(
    user      = 'RIOSLOPEZANDREA',
    password  = '',   # completar antes de ejecutar
    account   = 'elifweh-rg90787',
    warehouse = 'TFM_WH',
    database  = 'TFM_ECG',
    schema    = 'ANOMALY_DETECTION',
    role      = 'ACCOUNTADMIN'
)
cursor = conn.cursor()
print('Snowflake OK')

# ── Cliente S3 (para señales ECG crudas) ─────────────────
ACCESS_KEY = ''   # completar antes de ejecutar
SECRET_KEY = ''   # completar antes de ejecutar
BUCKET     = 'ptbxl-tfm-andrea'
PREFIX     = 'bronze/FileYear=2026/FileMonth=04/FileDay=13'

s3 = boto3.client('s3',
    aws_access_key_id     = ACCESS_KEY,
    aws_secret_access_key = SECRET_KEY,
    region_name           = 'eu-west-1'
)
print('S3 OK')

## F.3 — Vista 1 y Vista 2: Anomalías por modelo y por diagnóstico

In [ ]:
# Consulta 1: total de anomalías por modelo
cursor.execute("""
    SELECT inference_date, model_name, COUNT(*) as total_anomalias
    FROM ANOMALY_SCORES
    WHERE is_anomaly = TRUE
    GROUP BY inference_date, model_name
    ORDER BY inference_date
""")
df_temporal = pd.DataFrame(cursor.fetchall(),
    columns=['inference_date', 'model_name', 'total_anomalias'])

# Consulta 2: tasa de detección por diagnóstico (Isolation Forest)
cursor.execute("""
    SELECT m.diagnosis, COUNT(*) as total,
           SUM(CASE WHEN a.is_anomaly = TRUE THEN 1 ELSE 0 END) as anomalos
    FROM ANOMALY_SCORES a
    JOIN ECG_METADATA m ON a.ecg_id = m.ecg_id
    WHERE a.model_name = 'isolation_forest_v1'
    GROUP BY m.diagnosis
    ORDER BY anomalos DESC
    LIMIT 15
""")
df_diag = pd.DataFrame(cursor.fetchall(),
    columns=['diagnosis', 'total', 'anomalos'])
df_diag['tasa'] = (df_diag['anomalos'] / df_diag['total'] * 100).round(1)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Análisis de Anomalías ECG — PTB-XL', fontsize=14, fontweight='bold')

for model, grupo in df_temporal.groupby('model_name'):
    label = 'Isolation Forest' if 'isolation' in model else 'LSTM Autoencoder'
    axes[0].bar(label, grupo['total_anomalias'].values[0],
                color='#2E75B6' if 'isolation' in model else '#E74C3C', alpha=0.8)
axes[0].set_title('Total de Anomalías Detectadas por Modelo')
axes[0].set_ylabel('Cantidad de registros anómalos')
axes[0].grid(True, alpha=0.3, axis='y')

top10 = df_diag.head(10)
axes[1].barh(top10['diagnosis'], top10['tasa'], color='#2E75B6', alpha=0.8)
axes[1].set_title('Tasa de Detección por Diagnóstico\n(Isolation Forest — Top 10)')
axes[1].set_xlabel('Tasa de detección (%)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
display(fig)
plt.savefig('/tmp/viz_anomalias.png', dpi=150, bbox_inches='tight')
print('Vistas 1 y 2 generadas')

## F.4 — Vista 3: Señal ECG individual con error de reconstrucción LSTM

Se utiliza el ECG 3672 (diagnóstico: PACE), detectado como anómalo por el LSTM Autoencoder
con score 0.2834, superando el umbral de 0.0957. La morfología de un ECG con marcapasos
difiere significativamente de los patrones normales aprendidos durante el entrenamiento.

In [ ]:
# Consultar datos del ECG desde Snowflake
cursor.execute("""
    SELECT m.ecg_id, m.diagnosis, a.anomaly_score, a.threshold
    FROM ANOMALY_SCORES a
    JOIN ECG_METADATA m ON a.ecg_id = m.ecg_id
    WHERE a.ecg_id = 3672 AND a.model_name = 'lstm_autoencoder_v1'
""")
ecg_id, diagnosis, score, threshold = cursor.fetchone()
print(f'ECG {ecg_id} | Diagnostico: {diagnosis} | Score: {score:.4f} | Umbral: {threshold:.4f}')

# Cargar modelo LSTM desde S3
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, latent_size=32, num_layers=2):
        super(LSTMAutoencoder, self).__init__()
        self.encoder = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                               num_layers=num_layers, batch_first=True, dropout=0.2)
        self.latent        = nn.Linear(hidden_size, latent_size)
        self.decoder_input = nn.Linear(latent_size, hidden_size)
        self.decoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size,
                               num_layers=num_layers, batch_first=True, dropout=0.2)
        self.output_layer  = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        _, (hidden, _) = self.encoder(x)
        encoded    = self.latent(hidden[-1])
        dec_input  = self.decoder_input(encoded).unsqueeze(1).repeat(1, seq_len, 1)
        decoded, _ = self.decoder(dec_input)
        return self.output_layer(decoded), encoded

device = torch.device('cpu')
modelo_lstm = LSTMAutoencoder().to(device)
obj = s3.get_object(Bucket=BUCKET, Key='gold/models/lstm_autoencoder_v1.pkl')
model_data = pickle.loads(obj['Body'].read())
modelo_lstm.load_state_dict(model_data['modelo_state'])
modelo_lstm.eval()
print('Modelo LSTM cargado OK')

# Leer señal desde S3
local_base = '/tmp/03672_lr'
for ext in ['.dat', '.hea']:
    s3.download_file(BUCKET, f'{PREFIX}/records100/03000/03672_lr{ext}', f'{local_base}{ext}')

record     = wfdb.rdrecord(local_base)
signal_raw = record.p_signal[:1000, 0]

# Normalizar entre -1 y 1
s_min, s_max = signal_raw.min(), signal_raw.max()
signal_norm  = (2 * (signal_raw - s_min) / (s_max - s_min) - 1).astype('float32')

# Reconstrucción LSTM
with torch.no_grad():
    tensor     = torch.FloatTensor(signal_norm).unsqueeze(0).unsqueeze(-1)
    output, _  = modelo_lstm(tensor)
    signal_rec = output.squeeze().numpy()

# Error punto a punto
error_pp    = (signal_norm - signal_rec) ** 2
umbral_lstm = float(threshold)

# Visualización
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle(f'ECG {ecg_id} — Diagnostico: {diagnosis} | Score: {score:.4f} | Umbral: {umbral_lstm:.4f}',
             fontsize=13, fontweight='bold')

t = np.arange(len(signal_norm))

axes[0].plot(t, signal_norm, color='#2E75B6', linewidth=0.8, label='Señal original')
axes[0].set_title('Señal ECG original (Derivación I, normalizada)')
axes[0].set_ylabel('Amplitud')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, signal_rec, color='#E74C3C', linewidth=0.8, label='Reconstrucción LSTM')
axes[1].plot(t, signal_norm, color='#2E75B6', linewidth=0.4, alpha=0.4, label='Original')
axes[1].set_title('Señal reconstruida por el LSTM Autoencoder')
axes[1].set_ylabel('Amplitud')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].fill_between(t, error_pp, alpha=0.7, color='#E67E22', label='Error de reconstrucción')
axes[2].axhline(y=umbral_lstm, color='red', linestyle='--', linewidth=1.5,
                label=f'Umbral = {umbral_lstm:.4f}')
axes[2].set_title('Error de reconstrucción punto a punto (MSE)')
axes[2].set_ylabel('Error cuadrático')
axes[2].set_xlabel('Muestras (100 Hz x 10 segundos)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
display(fig)
plt.savefig('/tmp/viz_ecg_individual.png', dpi=150, bbox_inches='tight')
print('Vista 3 generada')

cursor.close()
conn.close()
print('Conexion Snowflake cerrada')